<a href="https://colab.research.google.com/github/main-polyboly/CienciaDeDatos-GestiondeCitas/blob/main/MIS_CITAMED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏥 CITAMED 2.0 — Sistema de Información Gerencial (MIS)
### Dashboard de KPIs Clínicos y Financieros
**Curso:** ISID223 Introducción a los Sistemas de Información  
**Objetivo:** Transformar datos del TPS (citas, facturas, recetas) en información gerencial accionable.


## Fase 3: ETL — Extracción, Transformación y Carga

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# ── Estilo global ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d2e',
    'axes.edgecolor':   '#2d3561',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#a0a0a0',
    'ytick.color':      '#a0a0a0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2d3561',
    'grid.linewidth':   0.5,
    'font.family':      'DejaVu Sans',
    'font.size':        10,
})
ACCENT  = ['#4fc3f7','#81c784','#ffb74d','#f06292','#ba68c8','#4db6ac','#ff8a65','#a5d6a7','#90caf9','#fff176']
BLUE    = '#4fc3f7'
GREEN   = '#81c784'
ORANGE  = '#ffb74d'
RED     = '#ef5350'
PURPLE  = '#ba68c8'

# ── Extracción ─────────────────────────────────────────────────
BASE = '/home/claude/'
citas    = pd.read_csv(BASE + 'citas.csv')
facturas = pd.read_csv(BASE + 'facturas.csv')
medicos  = pd.read_csv(BASE + 'medicos.csv')
pac      = pd.read_csv(BASE + 'pacientes.csv')

# ── Transformación ─────────────────────────────────────────────
citas['Fecha']         = pd.to_datetime(citas['Fecha'])
citas['Mes']           = citas['Fecha'].dt.to_period('M').astype(str)
facturas['Fecha_Cita'] = pd.to_datetime(facturas['Fecha_Cita'])
facturas['Mes']        = facturas['Fecha_Cita'].dt.to_period('M').astype(str)

# KPIs base
total_citas     = len(citas)
completadas     = (citas['Estado'] == 'Completada').sum()
canceladas      = (citas['Estado'] == 'Cancelada').sum()
tasa_comp       = completadas / total_citas * 100
tasa_canc       = canceladas  / total_citas * 100

total_recaudado = facturas[facturas['Estado_Pago']=='Pagada']['Total'].sum()
total_pendiente = facturas[facturas['Estado_Pago']=='Pendiente']['Total'].sum()
ticket_prom     = facturas[facturas['Estado_Pago']=='Pagada']['Total'].mean()
n_transacciones = (facturas['Estado_Pago']=='Pagada').sum()

print("✅ ETL completado")
print(f"   Citas:          {total_citas}")
print(f"   Completadas:    {completadas} ({tasa_comp:.1f}%)")
print(f"   Canceladas:     {canceladas}  ({tasa_canc:.1f}%)")
print(f"   Recaudado:      ${total_recaudado:,.2f}")
print(f"   Ticket promedio:${ticket_prom:.2f}")
print(f"   Transacciones:  {n_transacciones}")


## KPI Dashboard — Indicadores Clave de Desempeño

In [ ]:
# ═══════════════════════════════════════════════════
#  DASHBOARD 1: PANEL EJECUTIVO (KPIs 1, 2, 3, 4)
# ═══════════════════════════════════════════════════
fig = plt.figure(figsize=(18, 14), facecolor='#0f1117')
fig.suptitle('CITAMED 2.0 — Panel Ejecutivo MIS', fontsize=18, fontweight='bold',
             color='white', y=0.98)

gs = GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.4)

# ── KPI 1: Tarjetas métricas ──────────────────────────
metricas = [
    ('Total Citas',        total_citas,            BLUE,   ''),
    ('Citas Completadas',  completadas,             GREEN,  f'{tasa_comp:.1f}%'),
    ('Tasa Cancelación',   f'{tasa_canc:.1f}%',    RED,    f'{canceladas} citas'),
    ('Ticket Promedio',    f'${ticket_prom:.2f}',  ORANGE, 'por consulta'),
]
for i, (titulo, valor, color, sub) in enumerate(metricas):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor('#1a1d2e')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.axis('off')
    # borde de color
    for spine in ['top','bottom','left','right']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_color(color)
        ax.spines[spine].set_linewidth(2.5)
    ax.text(0.5, 0.72, str(valor), ha='center', va='center',
            fontsize=22, fontweight='bold', color=color)
    ax.text(0.5, 0.42, titulo, ha='center', va='center',
            fontsize=9, color='#a0a0a0')
    if sub:
        ax.text(0.5, 0.18, sub, ha='center', va='center',
                fontsize=8, color='#707070')

# ── KPI 2: Citas por estado (donut) ──────────────────
ax2 = fig.add_subplot(gs[1, 0:2])
estados = citas['Estado'].value_counts()
colores = [GREEN if e=='Completada' else RED if e=='Cancelada' else BLUE if e=='Confirmada' else ORANGE
           for e in estados.index]
wedges, texts, autotexts = ax2.pie(
    estados.values, labels=estados.index, colors=colores,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'width':0.55, 'edgecolor':'#0f1117', 'linewidth':2},
    textprops={'fontsize':9, 'color':'white'})
for at in autotexts: at.set_fontsize(8)
ax2.set_title('KPI 1 — Distribución de Citas por Estado', color='white', fontsize=11, pad=10)
ax2.set_facecolor('#1a1d2e')

# ── KPI 3: Ingresos por especialidad (barras horizontales) ───
ax3 = fig.add_subplot(gs[1, 2:4])
ing_esp = facturas[facturas['Estado_Pago']=='Pagada'].groupby('Especialidad')['Total'].sum().sort_values()
bars = ax3.barh(range(len(ing_esp)), ing_esp.values, color=ACCENT[:len(ing_esp)], alpha=0.85, height=0.7)
ax3.set_yticks(range(len(ing_esp)))
ax3.set_yticklabels([e[:18] for e in ing_esp.index], fontsize=8)
ax3.set_xlabel('Ingresos ($)', color='#a0a0a0')
ax3.set_title('KPI 3 — Ingresos por Especialidad', color='white', fontsize=11)
ax3.grid(axis='x', alpha=0.3)
for i, (v, b) in enumerate(zip(ing_esp.values, bars)):
    ax3.text(v + 50, i, f'${v:,.0f}', va='center', fontsize=7.5, color='#e0e0e0')

# ── KPI 4: Tendencia mensual de citas ─────────────────────────
ax4 = fig.add_subplot(gs[2, 0:2])
tend = citas.groupby('Mes').size().reset_index(name='Citas')
tend = tend.sort_values('Mes').tail(12)
ax4.fill_between(range(len(tend)), tend['Citas'], alpha=0.3, color=BLUE)
ax4.plot(range(len(tend)), tend['Citas'], 'o-', color=BLUE, linewidth=2, markersize=5)
ax4.set_xticks(range(len(tend)))
ax4.set_xticklabels([m[-5:] for m in tend['Mes']], rotation=45, fontsize=8)
ax4.set_ylabel('Nro. Citas')
ax4.set_title('KPI 4 — Tendencia Mensual de Citas', color='white', fontsize=11)
ax4.grid(alpha=0.3)

# ── KPI 5: Ingresos mensuales (barras) ────────────────────────
ax5 = fig.add_subplot(gs[2, 2:4])
ing_mes = facturas[facturas['Estado_Pago']=='Pagada'].groupby('Mes')['Total'].sum().reset_index()
ing_mes = ing_mes.sort_values('Mes').tail(12)
cols_bar = [GREEN if v >= ing_mes['Total'].mean() else ORANGE for v in ing_mes['Total']]
ax5.bar(range(len(ing_mes)), ing_mes['Total'], color=cols_bar, alpha=0.85)
ax5.axhline(ing_mes['Total'].mean(), color='white', linestyle='--', linewidth=1, alpha=0.5, label='Promedio')
ax5.set_xticks(range(len(ing_mes)))
ax5.set_xticklabels([m[-5:] for m in ing_mes['Mes']], rotation=45, fontsize=8)
ax5.set_ylabel('Ingresos ($)')
ax5.set_title('KPI 5 — Ingresos Mensuales', color='white', fontsize=11)
ax5.legend(fontsize=8)
ax5.grid(axis='y', alpha=0.3)

plt.savefig('/home/claude/dashboard_ejecutivo.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()
print("✅ Dashboard Ejecutivo generado")


In [ ]:
# ═══════════════════════════════════════════════════
#  DASHBOARD 2: ANÁLISIS OPERACIONAL (KPIs 6, 7, 8)
# ═══════════════════════════════════════════════════
fig2, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor='#0f1117')
fig2.suptitle('CITAMED 2.0 — Análisis Operacional', fontsize=16, fontweight='bold',
              color='white', y=1.01)

# ── KPI 6: Top 5 médicos por ingresos ─────────────────────────
ax = axes[0, 0]
top_med = facturas[facturas['Estado_Pago']=='Pagada'].groupby('Medico_Username')['Total'].sum()\
          .sort_values(ascending=False).head(5)
top_med_names = top_med.index.tolist()
ax.bar(range(len(top_med)), top_med.values, color=ACCENT[:5], alpha=0.85)
ax.set_xticks(range(len(top_med)))
ax.set_xticklabels([n.replace('dr','Dr.').capitalize() for n in top_med_names], rotation=20, fontsize=8)
ax.set_ylabel('Ingresos ($)')
ax.set_title('KPI 6 — Top 5 Médicos\npor Ingresos', color='white', fontsize=10)
ax.grid(axis='y', alpha=0.3)
for i, v in enumerate(top_med.values):
    ax.text(i, v + 50, f'${v:,.0f}', ha='center', fontsize=7.5, color='white')

# ── KPI 7: Método de pago ─────────────────────────────────────
ax = axes[0, 1]
metodo = facturas[facturas['Estado_Pago']=='Pagada']['MetodoPago'].value_counts()
ax.pie(metodo.values, labels=metodo.index, colors=ACCENT[:len(metodo)],
       autopct='%1.1f%%', startangle=140,
       wedgeprops={'edgecolor':'#0f1117', 'linewidth':1.5},
       textprops={'fontsize':8, 'color':'white'})
ax.set_title('KPI 7 — Métodos de Pago', color='white', fontsize=10)
ax.set_facecolor('#1a1d2e')

# ── KPI 8: Pacientes recurrentes (top 10 por citas) ───────────
ax = axes[0, 2]
top_pac = citas[citas['Estado']=='Completada']['Nombre_Paciente'].value_counts().head(8)
ax.barh(range(len(top_pac)), top_pac.values, color=PURPLE, alpha=0.8)
ax.set_yticks(range(len(top_pac)))
ax.set_yticklabels([n.split()[0] + ' ' + n.split()[-1] for n in top_pac.index], fontsize=8)
ax.set_xlabel('Consultas completadas')
ax.set_title('KPI 8 — Top Pacientes\nRecurrentes', color='white', fontsize=10)
ax.grid(axis='x', alpha=0.3)

# ── Motivos de cancelación ────────────────────────────────────
ax = axes[1, 0]
cancel = citas[citas['Estado']=='Cancelada']['MotivoCancel'].value_counts()
if len(cancel) > 0:
    ax.bar(range(len(cancel)), cancel.values, color=RED, alpha=0.7)
    ax.set_xticks(range(len(cancel)))
    ax.set_xticklabels(cancel.index, rotation=20, fontsize=8)
    ax.set_ylabel('Cantidad')
ax.set_title('Motivos de Cancelación', color='white', fontsize=10)
ax.grid(axis='y', alpha=0.3)

# ── Citas por hora del día ────────────────────────────────────
ax = axes[1, 1]
horas_dist = citas['Hora'].value_counts().sort_index()
ax.bar(horas_dist.index, horas_dist.values, color=BLUE, alpha=0.8)
ax.set_xlabel('Hora')
ax.set_ylabel('Citas')
ax.set_title('Citas por Hora del Día', color='white', fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=45)

# ── Estado de cartera (pagadas vs pendientes) ─────────────────
ax = axes[1, 2]
cartera = [
    facturas[facturas['Estado_Pago']=='Pagada']['Total'].sum(),
    facturas[facturas['Estado_Pago']=='Pendiente']['Total'].sum(),
    facturas[facturas['Estado_Pago']=='Anulada']['Total'].sum() if 'Anulada' in facturas['Estado_Pago'].values else 0
]
labels = ['Pagadas', 'Pendientes', 'Anuladas']
colores2 = [GREEN, ORANGE, RED]
bars2 = ax.bar(labels, cartera, color=colores2, alpha=0.85)
ax.set_ylabel('Monto ($)')
ax.set_title('Estado de Cartera', color='white', fontsize=10)
ax.grid(axis='y', alpha=0.3)
for b, v in zip(bars2, cartera):
    ax.text(b.get_x() + b.get_width()/2, v + 100, f'${v:,.0f}', ha='center', fontsize=8, color='white')

for row in axes:
    for a in row:
        a.set_facecolor('#1a1d2e')

plt.tight_layout()
plt.savefig('/home/claude/dashboard_operacional.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()
print("✅ Dashboard Operacional generado")


## Reporte Programado — Ventas del Mes Actual

In [ ]:
from datetime import date

mes_actual = date.today().strftime('%Y-%m')
df_mes = facturas[(facturas['Mes'] == mes_actual) & (facturas['Estado_Pago']=='Pagada')]
df_citas_mes = citas[citas['Mes'] == mes_actual]

print(f"{'='*55}")
print(f"  REPORTE MENSUAL — {mes_actual}")
print(f"{'='*55}")
print(f"  Citas agendadas        : {len(df_citas_mes)}")
print(f"  Citas completadas      : {(df_citas_mes['Estado']=='Completada').sum()}")
print(f"  Citas canceladas       : {(df_citas_mes['Estado']=='Cancelada').sum()}")
print(f"  Ingresos del mes       : ${df_mes['Total'].sum():,.2f}")
print(f"  Transacciones pagadas  : {len(df_mes)}")
print(f"  Ticket promedio del mes: ${df_mes['Total'].mean():.2f}" if len(df_mes) > 0 else "  Sin transacciones este mes.")
print(f"{'='*55}")

# Especialidad top del mes
if len(df_mes) > 0:
    top_esp_mes = df_mes.groupby('Especialidad')['Total'].sum().idxmax()
    print(f"  Especialidad líder     : {top_esp_mes}")
print(f"{'='*55}")


## Propuesta de Valor del MIS

| KPI | Descripción | Decisión Gerencial |
|-----|-------------|-------------------|
| 1 | Distribución de citas por estado | Detectar ineficiencias operativas |
| 2 | Ingresos por especialidad | Priorizar inversión en áreas rentables |
| 3 | Ticket promedio | Evaluar política de precios |
| 4 | Tendencia mensual de citas | Planificar capacidad y personal |
| 5 | Ingresos mensuales vs promedio | Control presupuestal |
| 6 | Top 5 médicos por ingresos | Reconocimiento y retención de talento |
| 7 | Métodos de pago preferidos | Optimizar canales de cobro |
| 8 | Pacientes recurrentes | Estrategia de fidelización |

### Ventaja Competitiva
El MIS transforma datos crudos del TPS en **inteligencia de negocio**: la gerencia pasa de operar *a ciegas* a tomar decisiones basadas en datos con actualización en tiempo real.
